In [1]:
# ================================================================
# URL MODEL CELL 1 — Imports
# ================================================================
import requests
from bs4 import BeautifulSoup
import pickle
import re
from urllib.parse import urlparse

print('Imports done!')

Imports done!


In [2]:
# ================================================================
# URL MODEL CELL 2 — Load Model
# ================================================================
vectorizer = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detector\news\AI\Text Models\Text Model\tfidf_vectorizer.pkl', 'rb'))
best_model = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detector\news\AI\Text Models\Text Model\best_fake_news_model.pkl', 'rb'))

print('Model loaded!')

Model loaded!


In [3]:
# ================================================================
# URL MODEL CELL 3 — Source Credibility Database
# ================================================================

# Trusted news sources — label 1 (credible)
TRUSTED_SOURCES = [
    'reuters.com', 'bbc.com', 'bbc.co.uk', 'apnews.com',
    'npr.org', 'theguardian.com', 'nytimes.com', 'washingtonpost.com',
    'bloomberg.com', 'economist.com', 'ft.com', 'wsj.com',
    'time.com', 'newsweek.com', 'theatlantic.com', 'politico.com',
    'cbsnews.com', 'nbcnews.com', 'abcnews.go.com', 'cnn.com',
    'aljazeera.com', 'dw.com', 'france24.com', 'euronews.com'
]

# Known fake/unreliable sources — label 0 (not credible)
FAKE_SOURCES = [
    'infowars.com', 'naturalnews.com', 'breitbart.com',
    'beforeitsnews.com', 'yournewswire.com', 'newslo.com',
    'empirenews.net', 'theonion.com', 'worldnewsdailyreport.com',
    'nationalreport.net', 'huzlers.com', 'superofficialnews.com',
    'abcnews.com.co', 'cbsnews.com.co', 'usatoday.com.co',
    'bbc.com.co', 'cnn.com.de', 'foxnews.com.co'
]

def check_source_credibility(url):
    try:
        # Extract domain from URL
        domain = urlparse(url).netloc
        # Remove www. prefix
        domain = domain.replace('www.', '')
        
        # Check against trusted sources
        for trusted in TRUSTED_SOURCES:
            if trusted in domain:
                return {
                    'domain'     : domain,
                    'status'     : 'Trusted ✅',
                    'score'      : 100,
                    'description': 'Known credible news source'
                }
        
        # Check against fake sources
        for fake in FAKE_SOURCES:
            if fake in domain:
                return {
                    'domain'     : domain,
                    'status'     : 'Unreliable ❌',
                    'score'      : 0,
                    'description': 'Known unreliable or fake news source'
                }
        
        # Unknown source
        return {
            'domain'     : domain,
            'status'     : 'Unknown ⚠️',
            'score'      : 40,
            'description': 'Source not in trusted or fake list — treat with caution'
        }
    
    except Exception as e:
        return {
            'domain'     : 'Unknown',
            'status'     : 'Error ⚠️',
            'score'      : 40,
            'description': f'Could not check source: {str(e)}'
        }

print('check_source_credibility() is ready!')

check_source_credibility() is ready!


In [4]:
# ================================================================
# URL MODEL CELL 4 — Scrape Article Text
# ================================================================
def scrape_article(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return None, f'Error: Could not access URL (status {response.status_code})'
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()
        
        article = soup.find('article')
        if article:
            text = article.get_text(separator=' ')
        else:
            paragraphs = soup.find_all('p')
            text = ' '.join([p.get_text() for p in paragraphs])
        
        text = ' '.join(text.split())
        
        if len(text) < 50:
            return None, 'Error: Could not extract enough text from this URL'
        
        return text, None
    
    except requests.exceptions.Timeout:
        return None, 'Error: URL took too long to respond'
    except requests.exceptions.ConnectionError:
        return None, 'Error: Could not connect to URL'
    except Exception as e:
        return None, f'Error: {str(e)}'

print('scrape_article() is ready!')

scrape_article() is ready!


In [5]:
# ================================================================
# URL MODEL CELL 5 — Predict Function
# ================================================================
def predict_news(text, confidence_threshold=0.40):
    text = re.sub(r'^[A-Z\s,\.]+\(Reuters\)\s*[-\u2013]\s*', '', str(text))
    text_tfidf = vectorizer.transform([text])
    if text_tfidf.nnz == 0:
        return 'Uncertain (No known words) ❓', 50
    prob = best_model.predict_proba(text_tfidf)[0]
    pred = best_model.predict(text_tfidf)[0]
    confidence = prob[pred]
    if confidence < confidence_threshold:
        return f'Uncertain (Confidence: {confidence:.2f}) ❓', 50
    label = 'Real 🟢' if pred == 1 else 'Fake 🔴'
    score = int(confidence * 100) if pred == 1 else int((1 - confidence) * 100)
    return label, score

print('predict_news() is ready!')

predict_news() is ready!


In [6]:
# ================================================================
# URL MODEL CELL 6 — Full Pipeline with Source Check
# ================================================================
def analyze_url(url):
    print(f'\n🔗 Analyzing: {url}')
    print('=' * 60)

    # Step 1 — Check source credibility
    source = check_source_credibility(url)
    print(f'📰 Source:      {source["domain"]}')
    print(f'🔍 Credibility: {source["status"]}')
    print(f'📝 Note:        {source["description"]}')
    print('=' * 60)

    # Step 2 — Scrape article text
    print('Fetching article text...')
    text, error = scrape_article(url)

    if error:
        print(error)
        # Still show source result even if scraping fails
        print(f'\nFinal Verdict based on source only:')
        print(f'Source: {source["status"]}')
        return

    print(f'✅ Text extracted ({len(text)} characters)')
    print(f'Preview: {text[:200]}...')
    print('=' * 60)

    # Step 3 — Predict from text
    text_prediction, text_score = predict_news(text)
    print(f'📊 Text Analysis:    {text_prediction}')
    print(f'📊 Source Check:     {source["status"]}')
    print('=' * 60)

    # Step 4 — Combined final verdict
    # Weight: 60% text model + 40% source credibility
    if 'Real' in text_prediction:
        text_credibility = text_score
    elif 'Fake' in text_prediction:
        text_credibility = 100 - text_score
    else:
        text_credibility = 50

    combined_score = int(0.6 * text_credibility + 0.4 * source['score'])

    if combined_score >= 70:
        final_verdict = 'Real 🟢'
        verdict_note  = 'Likely credible'
    elif combined_score >= 40:
        final_verdict = 'Uncertain ⚠️'
        verdict_note  = 'Treat with caution'
    else:
        final_verdict = 'Fake 🔴'
        verdict_note  = 'Likely not credible'

    print(f'\n{"=" * 60}')
    print(f'  FINAL VERDICT:  {final_verdict}')
    print(f'  Note:           {verdict_note}')
    print(f'  Combined Score: {combined_score}/100')
    print(f'{"=" * 60}')

    return {
        'url'            : url,
        'source'         : source,
        'text_prediction': text_prediction,
        'combined_score' : combined_score,
        'final_verdict'  : final_verdict
    }

print('analyze_url() is ready!')

analyze_url() is ready!


In [7]:
# ================================================================
# URL MODEL CELL 7 — Test it
# ================================================================

# ✅ Paste any news URL here
url = "https://www.reuters.com/world/us/senate-passes-government-funding-bill-2024-03-23/"

result = analyze_url(url)



🔗 Analyzing: https://www.reuters.com/world/us/senate-passes-government-funding-bill-2024-03-23/
📰 Source:      reuters.com
🔍 Credibility: Trusted ✅
📝 Note:        Known credible news source
Fetching article text...
Error: Could not access URL (status 401)

Final Verdict based on source only:
Source: Trusted ✅
